# Negative results kept on the record

Five things the gates rejected, kept because each one is a
demonstrated limit rather than a missing feature:

1. The gain of the tide transfer is invisible to binary wet/dry data.
   The affine degeneracy was hit twice independently (per-band α and
   the boundary's dominant-constituent gain), so the pipeline exposes
   it instead of printing a number it cannot know.
2. Separate flood/ebb clocks are not adoptable at this archive size.
   Three versions failed in three different ways — likelihood runaway,
   multiple-comparisons adoption under a judge with no margin, then
   insufficient power under the matched-null threshold — and each
   failure was caught by its control.
3. Censored (ponded) pixels refit at their spill level, erasing the
   evidence of their own censoring — a hard ceiling on any detector
   built from the refit map.
4. Submerged depth cannot be inverted from NDWI pixel by pixel in the
   never-exposed band: the bottom-albedo spread is larger than the
   depth signal.
5. A richer pixel model can win the fit and lose the product.

**You are here: 06.** The rejected ideas, kept with their controls — demonstrated limits, not missing features.

```text
+- the evidence chain ------------------------------------------------+
|  datacube -> water masks -> per-pixel wet/dry series                |
|    01 what is estimable  ->  02 boundary audit  ->  03 operator vs  |
|    gauges  ->  04 elevations vs truth  ->  05 uncertainty and       |
|    hydraulic layers  ->  06 negatives kept  ->  07 coast census     |
+---------------------------------------------------------------------+
```

In [1]:
import json
import os
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# run from the repo root so the results/ paths resolve
here = Path.cwd()
while not (here / "pyintertidal").is_dir():
    if here.parent == here:
        raise FileNotFoundError("repo root not found above " + str(Path.cwd()))
    here = here.parent
os.chdir(here)

def load(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

## Flood/ebb clocks under the null-thresholded judge

In [2]:
r = load("results/b5_gate_sim/veredicto_v3.json")
print("B5 v3 (null-thresholded adoption):", r["puerta"])
print(f"null adoption threshold (OOS improvement): "
      f"{r['umbral_nulo_delta_oos']:.4f}")
print("improvements measured with hysteresis planted:",
      np.round(r["mejoras_oos_con_histeresis"], 4))

B5 v3 (null-thresholded adoption): {'deteccion_ok': False, 'desdoble_ok': True, 'cota_ok': False, 'control_ok': True, 'PASA': False}
null adoption threshold (OOS improvement): 0.0101
improvements measured with hysteresis planted: [0.0003 0.0041 0.0061 0.0107 0.0002]


## The two degeneracies, measured

In [3]:
g = load("results/m2_gate_sim/result.json")["perfil_nll_alpha"]
print(f"NLL(alpha) range with scale-invariant grids: {g['recorrido']:.2e}")
b6 = load("results/b6_hydraulic_dem/result.json")["juez_simulacion"]
print(f"censoring self-detection against planted truth: "
      f"precision {b6['precision']:.2f}, recall {b6['exhaustividad']:.2f}")

NLL(alpha) range with scale-invariant grids: 0.00e+00
censoring self-detection against planted truth: precision 0.66, recall 0.40


## Blind-band depth inversion
The first design inverted every observation through the calibration
curve and hallucinated depth outside its monotone window — a bias
above a metre and a half, caught by the LiDAR grader. The second
design, restricted to low-tide scenes inside the window, removes the
hallucination and still has no skill against the trivial baseline
(assign every pixel the band middle):

In [4]:
# graded against LiDAR the estimator never saw; "trivial" assigns every
# pixel the band-middle elevation — the no-information baseline
r = load("results/p11b_blind_band/result.json")
print(f"never-exposed band, {r['n_px_estimados']:,} px: "
      f"RMSE {r['rmse_centrado']:.3f} m vs trivial {r['rmse_trivial']:.3f} m, "
      f"slope {r['pendiente']:.3f}, r {r['pearson']:.3f}")

never-exposed band, 28,204 px: RMSE 0.191 m vs trivial 0.169 m, slope 0.085, r 0.126


## The six-parameter pixel, gated out
Extending the pixel model with drainage and attenuation terms predicts
held-out NDWI better for a clear majority of pixels — and still
collapses the elevation slope against LiDAR. Third appearance of the
same compression trap, and the reason no model is adopted here on
residual RMSE alone:

In [5]:
r = load("results/p12_extended_pixel/result.json")
oos = r["oos_ndwi_rmse"]
print(f"OOS NDWI residual: 6-param better on {100*oos['frac_p6_better']:.0f}% "
      f"of pixels ({oos['p4']:.4f} -> {oos['p6']:.4f})")
print(f"z vs LiDAR slope: {r['z_vs_lidar']['p4']['pendiente']:.3f} -> "
      f"{r['z_vs_lidar']['p6']['pendiente']:.3f}")
print("gate:", r["gate"])

OOS NDWI residual: 6-param better on 67% of pixels (0.2303 -> 0.2282)
z vs LiDAR slope: 0.535 -> 0.330
gate: {'oos_majority': True, 'z_not_degraded': False, 'PASA': False}
